NOT USED IN FINAL

In [ ]:
import os
import json
import pandas as pd

RAW_ROOT = "RawData"
OUT_DIR = "team_data_processed"
os.makedirs(OUT_DIR, exist_ok=True)

records = []

for year in sorted(os.listdir(RAW_ROOT)):
    year_path = os.path.join(RAW_ROOT, year)
    if not os.path.isdir(year_path):
        continue

    for file in os.listdir(year_path):
        if not file.endswith(".json"):
            continue

        file_path = os.path.join(year_path, file)

        with open(file_path, "r", encoding="utf-8") as f:
            data = json.load(f)

        info = data.get("info", {})

        # ONLY T20 
        if info.get("match_type") != "T20":
            continue

        teams = info.get("teams", [])
        if len(teams) != 2:
            continue

        team1, team2 = teams
        match_id = os.path.splitext(file)[0]

        date = info.get("dates", [None])[0]

        toss = info.get("toss", {})
        toss_winner = toss.get("winner")
        toss_decision = toss.get("decision")

        # Bat first logic
        if toss_decision == "bat":
            bat_first = toss_winner
        elif toss_decision == "field":
            bat_first = team2 if toss_winner == team1 else team1
        else:
            bat_first = None

        outcome = info.get("outcome", {})
        winner = outcome.get("winner")

        # RESULT HANDLING
        if "result" in outcome:
          
            result_type = "no_result"
            result_team1_win = None

        elif winner is None:
            
            result_type = "tie"
            result_team1_win = 0.5

        else:
            result_type = "win"
            result_team1_win = 1.0 if winner == team1 else 0.0

        records.append({
            "match_id": match_id,
            "date": date,
            "team1": team1,
            "team2": team2,
            "toss_winner": toss_winner,
            "bat_first": bat_first,
            "winner": winner,
            "result_team1_win": result_team1_win,
            "result_type": result_type
        })

df = pd.DataFrame(records)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

df.to_csv(f"{OUT_DIR}/t20_match_metadata.csv", index=False)

print(f" Extracted {len(df)} T20 matches across all years")


✅ Extracted 8081 T20 matches across all years


manual merging of teams

In [ ]:


TEAM_OVERRIDE_MAP = {
    # IPL
    "Rising Pune Supergiant": "Rising Pune Supergiants",
    "Rising Pune Supergiants": "Rising Pune Supergiants",

    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Royal Challengers Bengaluru": "Royal Challengers Bengaluru",

    "Delhi Daredevils": "Delhi Capitals",
    "Delhi Capitals": "Delhi Capitals",

    "Kings XI Punjab": "Punjab Kings",
    "Punjab Kings": "Punjab Kings",

    "Sharjah Warriorz" :"Sharjah Warriors",
    "Sharjah Warriors":"Sharjah Warriors",


    # Example international
    "U.S.A.": "USA",
    "United States of America": "USA",
}

def normalize_team(name):
    return TEAM_OVERRIDE_MAP.get(name, name)

import pandas as pd

PATH = "team_data_processed/t20_match_metadata.csv"

df = pd.read_csv(PATH)

TEAM_COLS = ["team1", "team2", "toss_winner", "bat_first", "winner"]

for col in TEAM_COLS:
    if col in df.columns:
        df[col] = df[col].apply(
            lambda x: normalize_team(x) if pd.notna(x) else x
        )

df.to_csv(PATH, index=False)
print(" Team overrides applied (merged names)")


✅ Team overrides applied (merged names)


distinct team w ids

In [ ]:
import pandas as pd

MATCH_PATH = "team_data_processed/t20_match_metadata.csv"
MASTER_PATH = "team_data_processed/team_master.csv"

df = pd.read_csv(MATCH_PATH)

teams = sorted(pd.unique(df[["team1", "team2"]].values.ravel()))

def generate_unique_codes(team_names):
    used = set()
    codes = {}
    for name in team_names:
        base = name.replace("&", "").replace(".", "").replace(" ", "").upper()
        for l in [3, 4]:
            code = base[:l]
            if code not in used:
                break
        else:
            i = 2
            while f"{base[:3]}{i}" in used:
                i += 1
            code = f"{base[:3]}{i}"
        codes[name] = code
        used.add(code)
    return codes

codes = generate_unique_codes(teams)

team_master = pd.DataFrame({
    "team_id": range(1, len(teams) + 1),
    "team_name": teams,
    "team_code": [codes[t] for t in teams]
})

team_master.to_csv(MASTER_PATH, index=False)
print(" team_master rebuilt after merge")


✅ team_master rebuilt after merge


In [ ]:
import pandas as pd

MATCH_PATH = "team_data_processed/t20_match_metadata.csv"
MASTER_PATH = "team_data_processed/team_master.csv"

matches = pd.read_csv(MATCH_PATH, parse_dates=["date"])
teams = pd.read_csv(MASTER_PATH)

team_dates = pd.concat([
    matches[["team1", "date"]].rename(columns={"team1": "team"}),
    matches[["team2", "date"]].rename(columns={"team2": "team"})
])

last_match = (
    team_dates
    .groupby("team", as_index=False)["date"]
    .max()
    .rename(columns={"date": "last_match_date"})
)

teams = teams.merge(
    last_match,
    how="left",
    left_on="team_name",
    right_on="team"
).drop(columns=["team"])

teams.to_csv(MASTER_PATH, index=False)
print(" last_match_date added")


✅ last_match_date added


merge

In [11]:
matches = pd.read_csv("team_data_processed/t20_match_metadata.csv")
teams = pd.read_csv("team_data_processed/team_master.csv")

team_map = dict(zip(teams.team_name, teams.team_id))

matches["team1_id"] = matches["team1"].map(team_map)
matches["team2_id"] = matches["team2"].map(team_map)

matches.to_csv("team_data_processed/t20_match_metadata.csv", index=False)
